In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from wordcloud import WordCloud
import os

# Set style for all plots
sns.set_style("whitegrid")
plt.rcParams['figure.dpi'] = 100
plt.rcParams['savefig.dpi'] = 300
custom_palette = sns.color_palette("husl", 8)
sns.set_palette(custom_palette)

# Create output directory if it doesn't exist
output_dir = '/root/projects/portfolio_projects/data_analytics/civic_technology_project/data/images'
os.makedirs(output_dir, exist_ok=True)

# Load cleaned data
df = pd.read_csv('/root/projects/portfolio_projects/data_analytics/civic_technology_project/data/cleaned_talent_survey.csv', 
                 parse_dates=['Submission_Date'])

# 1. TEMPORAL ANALYSIS 
plt.figure(figsize=(14, 6))
daily_counts = df['Submission_Date'].value_counts().sort_index()

# Create line plot with improved formatting
ax = daily_counts.plot(kind='line', marker='o', linewidth=2, markersize=8, 
                      color=custom_palette[0], figsize=(14, 6))

# Add annotations for peaks
peak_date = daily_counts.idxmax().strftime('%b %d')
peak_value = daily_counts.max()
ax.annotate(f'Peak: {peak_value} responses\n{peak_date}', 
            xy=(daily_counts.idxmax(), peak_value),
            xytext=(10, 10), textcoords='offset points',
            bbox=dict(boxstyle='round,pad=0.5', fc='yellow', alpha=0.5),
            arrowprops=dict(arrowstyle='->'))

plt.title('Survey Response Trends Over Time', fontsize=14, pad=20)
plt.ylabel('Daily Responses', fontsize=12)
plt.xlabel('Date', fontsize=12)
plt.grid(True, alpha=0.3)
sns.despine()
plt.tight_layout()
plt.savefig(f'{output_dir}/01_survey_response_trends_over_time.png', bbox_inches='tight')
plt.show()

# 2. SATISFACTION ANALYSIS 
fig, ax = plt.subplots(1, 2, figsize=(16, 6), sharey=True)

# Lecture Satisfaction - Ordered bar plot with percentages
sns.countplot(data=df, x='Lecture_Satisfaction_Rating', ax=ax[0], 
              order=sorted(df['Lecture_Satisfaction_Rating'].unique()),
              saturation=0.8)

# Add percentage labels
total = len(df)
for p in ax[0].patches:
    height = p.get_height()
    ax[0].text(p.get_x() + p.get_width()/2., height + 3,
               f'{height/total:.1%}',
               ha='center', fontsize=10)

ax[0].set_title('Lecture Satisfaction Distribution', fontsize=13, pad=15)
ax[0].set_xlabel('Rating (1-5)', fontsize=11)
ax[0].set_ylabel('Count', fontsize=11)

# Course Relevance - Cleaned and formatted
df['Course_Relevance_To_Industry_Rating'] = df['Course_Relevance_To_Industry_Rating'].replace({
    '1': '1.0', '2': '2.0', '3': '3.0', '4': '4.0', '5': '5.0'
})

sns.countplot(data=df, x='Course_Relevance_To_Industry_Rating', ax=ax[1],
              order=['No Response', '1.0', '2.0', '3.0', '4.0', '5.0'],
              saturation=0.8)

# Add percentage labels
for p in ax[1].patches:
    height = p.get_height()
    ax[1].text(p.get_x() + p.get_width()/2., height + 3,
               f'{height/total:.1%}',
               ha='center', fontsize=10)

ax[1].set_title('Perceived Course Relevance to Industry', fontsize=13, pad=15)
ax[1].set_xlabel('Rating (1-5)', fontsize=11)
ax[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.savefig(f'{output_dir}/02_lecture_satisfaction_and_course_relevance.png', bbox_inches='tight')
plt.show()

# 3. DEMOGRAPHIC BREAKDOWN 
fig, ax = plt.subplots(1, 2, figsize=(18, 6))

# Age and Gender - Horizontal bar plot
age_gender = df.groupby(['Age_Group', 'Gender']).size().unstack()
age_gender.plot(kind='barh', stacked=True, ax=ax[0], width=0.8)
ax[0].set_title('Response Distribution by Age and Gender', fontsize=13, pad=15)
ax[0].set_xlabel('Number of Responses', fontsize=11)
ax[0].set_ylabel('Age Group', fontsize=11)
ax[0].legend(title='Gender', bbox_to_anchor=(1, 1))

# Response Distribution by Institution and Gender (Count Plot)
plt.sca(ax[1])
sns.countplot(data=df, x='Institution_Type', hue='Gender', 
              order=df['Institution_Type'].value_counts().index,
              palette=custom_palette[2:4], ax=ax[1])
ax[1].set_title('Response Distribution by Institution and Gender', fontsize=13, pad=15)
ax[1].set_xlabel('Institution Type', fontsize=11)
ax[1].set_ylabel('Number of Responses', fontsize=11)
ax[1].legend(title='Gender', bbox_to_anchor=(1, 1))
ax[1].tick_params(axis='x', rotation=45)

# Add value labels
for p in ax[1].patches:
    height = p.get_height()
    if height > 0:
        ax[1].text(p.get_x() + p.get_width()/2., height + 0.5,
                  f'{int(height)}',
                  ha='center', va='bottom', fontsize=10)

plt.tight_layout()
plt.savefig(f'{output_dir}/03_demographic_breakdown_age_gender_institution.png', bbox_inches='tight')
plt.show()

# 4. PROGRAMMING LANGUAGE ANALYSIS
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 8))

# Word Cloud
lang_data = df['Programming_Languages_In_Curriculum'].dropna().str.split(',\s*')
all_langs = [lang.strip() for sublist in lang_data for lang in sublist]
wordcloud = WordCloud(width=800, height=500, background_color='white',
                     colormap='viridis', max_words=50).generate(' '.join(all_langs))
ax1.imshow(wordcloud, interpolation='bilinear')
ax1.set_title('Programming Language Frequency (Word Cloud)', fontsize=14, pad=20)
ax1.axis('off')

# Bar Chart of Top Languages
lang_counts = pd.Series(all_langs).value_counts()
top_langs = lang_counts.head(15)
sns.barplot(x=top_langs.values, y=top_langs.index, ax=ax2, palette='viridis')
ax2.set_title('Top 15 Programming Languages in Curriculum', fontsize=14, pad=20)
ax2.set_xlabel('Frequency', fontsize=12)
ax2.set_ylabel('Language', fontsize=12)

# Add value labels
for i, v in enumerate(top_langs.values):
    ax2.text(v + 0.5, i, str(v), color='black', ha='left', va='center')

plt.tight_layout()
plt.savefig(f'{output_dir}/04_programming_language_analysis_wordcloud_barchart.png', bbox_inches='tight')
plt.show()

print(f"All visualizations saved to: {output_dir}")

In [ ]:
# This notebook performs hypothesis testing and visualization on the cleaned Civic Technology Talent Survey dataset.

# 1. Imports and Configuration
import pandas as pd
import numpy as np
from scipy import stats
import matplotlib.pyplot as plt
import seaborn as sns
import os

# Set visual aesthetics
sns.set_style("whitegrid")
plt.rcParams['figure.dpi'] = 100

# Define image path
img_dir = '/root/projects/portfolio_projects/data_analytics/civic_technology_project/data/images'
os.makedirs(img_dir, exist_ok=True)

# Mapping dictionaries for legends
satisfaction_labels = {
    1: "Very Dissatisfied",
    2: "Dissatisfied",
    3: "Indifferent",
    4: "Satisfied",
    5: "Very Satisfied"
}

relevance_labels = {
    1: "Strongly Disagree",
    2: "Disagree",
    3: "Neutral",
    4: "Agree",
    5: "Strongly Agree"
}


# 2. Load Dataset
df = pd.read_csv('/root/projects/portfolio_projects/data_analytics/civic_technology_project/data/cleaned_talent_survey.csv')


# 3. Industrial Training Sessions: One-Sample T-Test
print("=== INDUSTRIAL TRAINING SESSIONS ANALYSIS ===")
trainings = df['Number_of_Industrial_Trainings_Required'].dropna()

t_stat, p_val = stats.ttest_1samp(trainings, popmean=2)
print(f"\n1. T-test for Industrial Trainings = 2:")
print(f"• Mean: {trainings.mean():.2f}")
print(f"• T-statistic: {t_stat:.3f}")
print(f"• P-value: {p_val:.4f}")
print("• Conclusion:", 
      "Reject null hypothesis - average is significantly different from 2" if p_val < 0.05 
      else "Fail to reject null hypothesis - no significant difference from 2")

# Visualization
plt.figure(figsize=(10, 5))
sns.histplot(trainings, bins=5, kde=True)
plt.axvline(2, color='red', linestyle='--', label='Hypothesized mean (2)')
plt.title('Distribution of Required Industrial Training Sessions')
plt.xlabel('Number of Trainings')
plt.ylabel('Count')
plt.legend()
plt.tight_layout()
plt.savefig(f"{img_dir}/distribution_industrial_trainings.png")
plt.show()


# 4. Lecture Satisfaction: Wilcoxon Signed-Rank Test
print("\n=== LECTURE SATISFACTION ANALYSIS ===")
satisfaction = df['Lecture_Satisfaction_Rating'].dropna().astype(int)
satisfaction = satisfaction[satisfaction.isin([1, 2, 3, 4, 5])]

diffs = satisfaction - 3
diffs = diffs[diffs != 0]

w_stat, p_val = stats.wilcoxon(diffs)
print(f"\n2. Wilcoxon Signed-Rank Test for Satisfaction ≠ 3:")
print(f"• Median Rating: {satisfaction.median()}")
print(f"• Wilcoxon Statistic: {w_stat}")
print(f"• P-value: {p_val:.4f}")
print("• Conclusion:", 
      "Reject null hypothesis - median satisfaction is significantly different from 'Indifferent'" if p_val < 0.05 
      else "Fail to reject null hypothesis - no significant difference from 'Indifferent'")

# Visualization
plt.figure(figsize=(10, 5))
ax = sns.countplot(x=satisfaction, order=[1, 2, 3, 4, 5], palette='coolwarm')
plt.title('Distribution of Lecture Satisfaction Ratings')
plt.xlabel('Rating (1=Very Dissatisfied to 5=Very Satisfied)')
plt.ylabel('Count')

# Add legend
handles = [plt.Line2D([0], [0], marker='o', linestyle='', label=f"{i} = {satisfaction_labels[i]}", 
                      markersize=8, color='gray') for i in range(1, 6)]
plt.legend(title="Rating Scale", handles=handles, bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.savefig(f"{img_dir}/lecture_satisfaction_distribution.png")
plt.show()


# 5. Course Relevance: Chi-Square Goodness-of-Fit Test
print("\n=== COURSE RELEVANCE ANALYSIS (CHI-SQUARE) ===")
relevance = pd.to_numeric(df['Course_Relevance_To_Industry_Rating'], errors='coerce')
relevance = relevance.dropna().astype(int)
relevance = relevance[relevance.isin([1, 2, 3, 4, 5])]

observed_freq = relevance.value_counts().sort_index()
print("Observed agreement ratings:")
print(observed_freq)

expected_freq = np.full_like(observed_freq, observed_freq.sum() / len(observed_freq), dtype=float)

chi2_stat, p_val = stats.chisquare(f_obs=observed_freq, f_exp=expected_freq)
print(f"\n3. Chi-square Test for Perceptions of Course Relevance:")
print(f"• Chi-square statistic: {chi2_stat:.3f}")
print(f"• P-value: {p_val:.4f}")
print("• Conclusion:",
      "Reject null hypothesis - perceptions are not equally distributed" if p_val < 0.05
      else "Fail to reject null hypothesis - no significant deviation from expected")

# Visualization
plt.figure(figsize=(10, 5))
ax = sns.barplot(x=observed_freq.index, y=observed_freq.values, palette="viridis")
plt.title('Perceptions of Course Relevance to Tech Industry')
plt.xlabel('Agreement Level (1=Strongly Disagree to 5=Strongly Agree)')
plt.ylabel('Count')

# Add legend
handles = [plt.Line2D([0], [0], marker='s', linestyle='', label=f"{i} = {relevance_labels[i]}", 
                      markersize=8, color='gray') for i in range(1, 6)]
plt.legend(title="Agreement Scale", handles=handles, bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.savefig(f"{img_dir}/course_relevance_distribution.png")
plt.show()
